# Fine-tuning NLLB (Fongbe ↔ Français) — Kaggle

Ce notebook regroupe le pipeline **qui a fonctionné** pour entraîner le traducteur XoNet sur Kaggle.

## Prérequis (côté Kaggle)
1. **Accelerator = GPU** (Settings → Hardware accelerator)
2. **Internet = On** (indispensable pour pip et le téléchargement du modèle)
3. **Add Data** : ajouter le dataset `cleaned_dataset_final.csv` et le script `finetune_nllb.py`

> Le script `finetune_nllb.py` est dans `src/` du dépôt XoNet. Il est patché ici pour tourner sur le GPU T4 sans dépasser la mémoire (fp16 + Adafactor + batch 4).

In [ ]:
# ============================================
# CELLULE 1 : Installation des dépendances
# ============================================
!pip install -q transformers datasets evaluate sentencepiece sacrebleu pandas tqdm

In [ ]:
# ============================================
# CELLULE 2 : PATCH du script + LANCEMENT
# ============================================
# Adapter les chemins ci-dessous au résultat de :
#   import os
#   for r, d, f in os.walk('/kaggle/input'):
#       for x in f: print(os.path.join(r, x))

import os, shutil

# --- 1) Copier et patcher le script ---
src = "/kaggle/input/datasets/victoriatovihouande/1-fine/finetune_nllb.py"
dst = "/kaggle/working/finetune_nllb.py"
shutil.copy(src, dst)

with open(dst, encoding="utf-8") as f:
    code = f.read()

# Patch 1 : charger le modèle en FP16 (mémoire /2)
ancien = '''    model = AutoModelForSeq2SeqLM.from_pretrained(
        MODEL_CONFIG["model_name"],
        cache_dir=MODEL_CONFIG["cache_dir"],
    )'''
nouveau = '''    model = AutoModelForSeq2SeqLM.from_pretrained(
        MODEL_CONFIG["model_name"],
        cache_dir=MODEL_CONFIG["cache_dir"],
        torch_dtype=torch.float16,
    )'''
assert ancien in code, "Patch 1 KO"
code = code.replace(ancien, nouveau)

# Patch 2 : optimiseur Adafactor (léger en mémoire)
ancien = '        report_to="none",'
nouveau = '''        report_to="none",
        optim="adafactor",
        optim_args="relative_step=False, scale_parameter=False, warmup_init=False",'''
assert ancien in code, "Patch 2 KO"
code = code.replace(ancien, nouveau, 1)

# Patch 3 : pas de GradScaler (le modèle est déjà en fp16)
ancien = '        fp16=(device.type == "cuda"),'
nouveau = '        fp16=False,'
assert ancien in code, "Patch 3 KO"
code = code.replace(ancien, nouveau)

with open(dst, "w", encoding="utf-8") as f:
    f.write(code)

print("\u2705 Script patch\u00e9 (fp16 + Adafactor + batch 4)")

# --- 2) Variables d'environnement ---
os.environ["XONET_DATA_PATH"] = "/kaggle/input/datasets/victoriatovihouande/xonet-dataset/cleaned_dataset_final.csv"
os.environ["XONET_OUTPUT_DIR"] = "/kaggle/working/models/nllb-finetuned-fon-fr"
os.environ["XONET_CHECKPOINT_DIR"] = "/kaggle/working/models/checkpoints"

# --- 3) Lancer l'entraînement (~7h sur T4) ---
!python /kaggle/working/finetune_nllb.py

## Une fois l'entraînement terminé

Lorsque tu vois « **Fine-tuning terminé avec succès !** », exécute la cellule suivante **avant de fermer** :
elle pousse le modèle vers un dataset Kaggle (l'upload continue en arrière-plan).

> ⚠️ Ne pas compter sur « Quick Save » : il ne sauvegarde que le code, pas le modèle.

In [ ]:
# ============================================
# CELLULE 3 : SAUVEGARDE -> dataset Kaggle
# ============================================
# N'upload que les fichiers utiles (pas les checkpoints, ~10 Go en trop).

import os, json, shutil

model_dir = "/kaggle/working/models/nllb-finetuned-fon-fr"
assert os.path.isdir(model_dir), "Mod\u00e8le introuvable !"

# Repartir d'un dossier propre
dest = "/kaggle/dataset"
if os.path.isdir(dest):
    shutil.rmtree(dest)
os.makedirs(dest, exist_ok=True)

# Copier uniquement les fichiers (on ignore les dossiers checkpoint-XXX)
for nom in os.listdir(model_dir):
    chemin = os.path.join(model_dir, nom)
    if os.path.isfile(chemin):
        shutil.copy(chemin, os.path.join(dest, nom))
        print("  \u2713", nom)

# Métadonnées (licence obligatoire)
with open(os.path.join(dest, "dataset-metadata.json"), "w") as f:
    json.dump({
        "id": "victoriatovihouande/xonet-nllb-fon-fr",
        "title": "XoNet NLLB fon-fr",
        "licenses": [{"name": "CC-BY-NC-4.0"}],
    }, f)

print("\u2705 Fichiers pr\u00eats, lancement de l'upload...")
# Créer le dataset (sans argument supplémentaire)
!kaggle datasets create -p /kaggle/dataset